# EDA — OFF Data Ontology Categories
Exploratory analysis of the `cat`, `cat_l1`, `cat_l2`, and `cat_l3` columns in `off_data_ontology.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120


## 1 — Load & Preview

In [ ]:
df = pd.read_csv("off_data_ontology.csv")
print(f"Shape : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)


## 2 — Null / Coverage Analysis

In [ ]:
CAT_COLS = ["cat", "cat_l1", "cat_l2", "cat_l3"]

null_summary = pd.DataFrame({
    "null_count":  df[CAT_COLS].isnull().sum(),
    "null_pct":    (df[CAT_COLS].isnull().mean() * 100).round(1),
    "unique_vals": df[CAT_COLS].nunique(),
})
print("Category column overview:")
print(null_summary.to_string())

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(null_summary.index, null_summary["null_pct"], color=sns.color_palette("muted"))
ax.set_xlabel("% null")
ax.set_title("Null % per category column")
for bar, val in zip(ax.patches, null_summary["null_pct"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## 3 — L1 Distribution

In [ ]:
l1_counts = df["cat_l1"].value_counts(dropna=False).rename_axis("cat_l1").reset_index(name="count")
l1_counts["pct"] = (l1_counts["count"] / len(df) * 100).round(1)
print(l1_counts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
sns.barplot(data=l1_counts[l1_counts["cat_l1"].notna()],
            x="count", y="cat_l1", ax=axes[0], palette="muted")
axes[0].set_title("Rows per L1 category")
axes[0].set_xlabel("Row count")
axes[0].set_ylabel("")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width()):,}", va="center", fontsize=8)

# Pie chart (excluding nulls)
pie_data = l1_counts[l1_counts["cat_l1"].notna()]
axes[1].pie(pie_data["count"], labels=pie_data["cat_l1"],
            autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("muted", len(pie_data)))
axes[1].set_title("L1 category share")

plt.tight_layout()
plt.show()


## 4 — L2 Distribution (top 30)

In [ ]:
l2_counts = df["cat_l2"].value_counts(dropna=False).head(30).rename_axis("cat_l2").reset_index(name="count")

fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(data=l2_counts[l2_counts["cat_l2"].notna()],
            x="count", y="cat_l2", ax=ax, palette="muted")
ax.set_title("Top 30 L2 categories")
ax.set_xlabel("Row count")
ax.set_ylabel("")
for bar in ax.patches:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width()):,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()


## 5 — L2 breakdown within each L1

In [ ]:
df_mapped = df[df["cat_l1"].notna() & df["cat_l2"].notna()].copy()
l1_l2 = (df_mapped.groupby(["cat_l1", "cat_l2"])
         .size().reset_index(name="count")
         .sort_values(["cat_l1", "count"], ascending=[True, False]))

l1_list = sorted(l1_l2["cat_l1"].unique())
ncols = 3
nrows = -(-len(l1_list) // ncols)   # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
axes = axes.flatten()

for ax, l1 in zip(axes, l1_list):
    sub = l1_l2[l1_l2["cat_l1"] == l1].head(8)
    sns.barplot(data=sub, x="count", y="cat_l2", ax=ax, palette="muted")
    ax.set_title(l1, fontsize=10, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# hide unused axes
for ax in axes[len(l1_list):]:
    ax.set_visible(False)

plt.suptitle("L2 breakdown within each L1", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 6 — Top 30 original `cat` values (L3) and how they map to L1

In [ ]:
top_cats = (df.groupby(["cat", "cat_l1"])
            .size().reset_index(name="count")
            .sort_values("count", ascending=False)
            .head(30))

fig, ax = plt.subplots(figsize=(11, 9))
palette = dict(zip(df["cat_l1"].dropna().unique(),
                   sns.color_palette("muted", df["cat_l1"].nunique())))
colors = [palette.get(l1, "grey") for l1 in top_cats["cat_l1"]]

ax.barh(top_cats["cat"], top_cats["count"], color=colors)
ax.invert_yaxis()
ax.set_title("Top 30 original `cat` values — coloured by L1")
ax.set_xlabel("Row count")

# legend
from matplotlib.patches import Patch
handles = [Patch(color=palette[l1], label=l1) for l1 in sorted(palette)]
ax.legend(handles=handles, title="L1", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()


## 7 — Unmapped rows: which original `cat` values have no L1?

In [ ]:
unmapped = df[df["cat_l1"].isna()].copy()
print(f"Unmapped rows  : {len(unmapped):,}  ({len(unmapped)/len(df)*100:.1f}%)")
print(f"Unique cat vals: {unmapped['cat'].nunique()}")

top_unmapped = unmapped["cat"].value_counts().head(25).rename_axis("cat").reset_index(name="count")

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top_unmapped, x="count", y="cat", ax=ax, color=sns.color_palette("muted")[3])
ax.set_title("Top 25 unmapped `cat` values (no L1 assigned)")
ax.set_xlabel("Row count")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 8 — L1 → L2 heatmap (co-occurrence counts)

In [ ]:
pivot = (df_mapped.groupby(["cat_l1", "cat_l2"])
         .size().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.4, ax=ax, annot_kws={"size": 7})
ax.set_title("L1 × L2 co-occurrence heatmap")
ax.set_xlabel("L2")
ax.set_ylabel("L1")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


## 9 — Category hierarchy: unique L3 values per L2

In [ ]:
l3_per_l2 = (df_mapped.groupby("cat_l2")["cat_l3"]
             .nunique().sort_values(ascending=False)
             .head(30).rename_axis("cat_l2").reset_index(name="unique_l3"))

fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(data=l3_per_l2, x="unique_l3", y="cat_l2", ax=ax, palette="muted")
ax.set_title("Unique L3 (original cat) values per L2 — top 30")
ax.set_xlabel("Unique L3 count")
ax.set_ylabel("")
for bar in ax.patches:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width())}", va="center", fontsize=8)
plt.tight_layout()
plt.show()
